# Lab 4: Customer Support Chatbot with Memory

**AI Agent Lab 4 of 8**

---

## What this lab does

In this lab you will build a rule-based Customer Support Chatbot. The bot answers Frequently Asked Questions (FAQs) about a fictional coffee shop - things like opening hours, location, refund policy, and contact details. More importantly, the bot has memory: it remembers the user's name after they introduce themselves, and it keeps a log of the entire conversation history in a list. Because we are in a notebook (not a terminal), the main demo runs on a hardcoded list of sample messages. There is also an optional interactive version using `input()` in a separate cell.

## What you will learn

**Python concepts:**
- How to use a dictionary as a knowledge base (FAQ lookup)
- How to search strings for keywords using `in` and `.lower()`
- How to use a variable as memory (storing the user's name)
- How to build a conversation history as a list of dictionaries
- How to use `input()` for interactive programs (optional cell)

**Agent concepts:**
- Short-term memory in chatbots
- Intent detection using keyword matching
- Personalisation (using the user's name in replies)
- Conversation history as context

## Important things to know before starting

- **What is a chatbot?** A chatbot is a program that holds a conversation by responding to text messages. Ours is rule-based, meaning it follows fixed rules rather than ML.
- **What is 'memory' in a chatbot?** Memory means the bot stores information from earlier in the conversation and uses it in later replies.
- **Why no input() loop in the main demo?** Jupyter notebooks are not interactive terminals. Looping on `input()` can freeze the notebook. We simulate conversation with a pre-written list and offer `input()` separately.
- **This lab runs completely offline.**

## How the agent works

```
User message arrives
        |
        v
  Think: detect intent (name introduction, FAQ, greeting, farewell, unknown)
        |
        v
  Act: look up answer in FAQ dictionary or fill in name from memory
        |
        v
  Remember: add message and reply to conversation history list
        |
        v
  Report: print the reply with the user's name if known
```

---
## Step 0: Libraries and Setup

This lab uses **only built-in Python libraries**.

We use:
- `re` - for flexible keyword pattern matching in user messages
- `datetime` - to timestamp each conversation turn
- `sys` - to print the Python version

In [ ]:
# No installation needed
# Optional: # !pip install nothing

import sys
print('Setup check passed!')
print(f'Python version: {sys.version}')

---
## Importing Libraries

| Library | What it is used for in this lab |
|---|---|
| `re` | Case-insensitive pattern matching for intent detection |
| `datetime` | Timestamp for each conversation turn |

In [ ]:
import re        # used to detect keywords in user messages
import datetime # used to timestamp conversation turns

print('Libraries imported successfully.')

---
## Step 1: Define the FAQ Knowledge Base

**What this block does:** We create a dictionary called `faq_knowledge_base`. Each key is an intent name (a string that describes what the user is asking), and each value is the bot's reply for that intent. We also define a list of keywords for each intent so the bot knows how to detect it.

**Why we need it:** The FAQ dictionary is the bot's memory of facts. It is easy to extend - just add new key-value pairs to add new topics.

**What to expect as output:** The list of supported intents is printed.

In [ ]:
# The FAQ knowledge base - maps intent names to bot reply text
# To add a new FAQ: add a new key-value pair here and add keywords in step 2
faq_knowledge_base = {
    'hours': (
        'We are open Monday to Friday from 7:30 AM to 9:00 PM, '
        'Saturday from 8:00 AM to 10:00 PM, and Sunday from 9:00 AM to 7:00 PM.'
    ),
    'location': (
        'We are located at 42 Greenfield Avenue, City Centre. '
        'We are next to the Central Park subway station, Exit B.'
    ),
    'refund': (
        'We offer a full refund within 14 days of purchase for any unopened item. '
        'For opened items, we offer store credit. Please bring your receipt.'
    ),
    'contact': (
        'You can reach us by email at support@coffeecorner.com, '
        'by phone at +1-555-COFFEE (263333), '
        'or by visiting us in person during business hours.'
    ),
    'menu': (
        'We serve espresso, cappuccino, flat white, cold brew, and seasonal specials. '
        'We also have pastries, sandwiches, and vegan options. '
        'Ask a barista for today\'s special!'
    ),
    'wifi': (
        'Yes, we have free WiFi! The network name is CoffeeCorner_Guest '
        'and the password is written on the back of your receipt.'
    )
}

print(f'FAQ knowledge base loaded with {len(faq_knowledge_base)} topics.')
print(f'Topics: {list(faq_knowledge_base.keys())}')

---
## Step 2: Define Intent Keywords

**What this block does:** We define a dictionary called `intent_keyword_map`. Each key is an intent name (matching the FAQ keys), and each value is a list of keywords that indicate the user is asking about that topic.

**Why we need it:** Separating the keywords from the answers makes the code cleaner. You can tune how the bot detects intents without touching the FAQ answers.

**What to expect as output:** A count of intent patterns.

In [ ]:
# Maps each intent to the keywords that trigger it.
# The bot searches user messages for these keywords (case-insensitive).
intent_keyword_map = {
    'hours':    ['hours', 'open', 'close', 'when', 'time', 'schedule'],
    'location': ['where', 'address', 'location', 'find you', 'directions', 'situated'],
    'refund':   ['refund', 'return', 'money back', 'exchange', 'receipt'],
    'contact':  ['contact', 'email', 'phone', 'call', 'reach'],
    'menu':     ['menu', 'food', 'drink', 'coffee', 'eat', 'serve', 'offer'],
    'wifi':     ['wifi', 'internet', 'password', 'network', 'connect']
}

print(f'Intent keyword map loaded with {len(intent_keyword_map)} intents.')

---
## Step 3: Build the Intent Detector and Memory

**What this block does:** We define two things:
1. `detect_intent()` - reads a user message and returns the matching intent name, or `'unknown'` if no keyword matches.
2. `extract_user_name()` - checks if the message looks like a name introduction (e.g., 'My name is Ahmed') and extracts the name.

We also set up the memory variables: `remembered_user_name` and `conversation_history_list`.

**What to expect as output:** The functions are defined and memory is initialised.

In [ ]:
# Memory variables - these persist across conversation turns
# remembered_user_name stores the name once the user introduces themselves
remembered_user_name = None  # None means we do not know the user's name yet

# conversation_history_list stores all turns as a list of dictionaries
# Each entry has: 'turn', 'user_said', 'bot_replied', 'timestamp'
conversation_history_list = []


def detect_intent(user_message_text):
    """
    Reads a user message and returns the matching intent name.

    Compares the message (lowercased) against every keyword in intent_keyword_map.

    Takes in:
      user_message_text - the message the user sent (string)

    Returns:
      The intent name (string) like 'hours', 'location', etc.,
      or 'greeting', 'farewell', or 'unknown' if no FAQ intent matches.
    """

    lowered_message = user_message_text.lower()

    # Check for a greeting first
    if any(word in lowered_message for word in ['hello', 'hi', 'hey', 'good morning', 'howdy']):
        return 'greeting'

    # Check for a farewell
    if any(word in lowered_message for word in ['bye', 'goodbye', 'see you', 'take care', 'thanks bye']):
        return 'farewell'

    # Check for name introduction
    if re.search(r'my name is|i am|i\'m|call me', lowered_message):
        return 'name_introduction'

    # Check each FAQ intent
    for intent_name, keyword_list in intent_keyword_map.items():
        for keyword in keyword_list:
            if keyword in lowered_message:
                return intent_name

    return 'unknown'


def extract_user_name(user_message_text):
    """
    Tries to extract the user's name from a message like 'My name is Ahmed'.

    Takes in:
      user_message_text - the message containing the name introduction (string)

    Returns:
      The extracted name (string), or None if no name could be found.
    """

    # Try to match patterns like 'my name is X', 'I am X', or 'call me X'
    name_match = re.search(
        r'(?:my name is|i am|i\'m|call me)\s+([A-Za-z]+)',
        user_message_text,
        re.IGNORECASE
    )
    if name_match:
        return name_match.group(1).capitalize()
    return None


print('Intent detector and memory functions defined.')
print(f'Memory initialised: user name = {remembered_user_name}, history = {conversation_history_list}')

---
## Step 4: Build the Chatbot Agent

**What this block does:** We define `customer_support_chatbot()`, the main bot function. It takes a user message, detects the intent, generates a personalised reply, updates the memory, and logs the turn in `conversation_history_list`.

**Why we need it:** This function ties together intent detection, memory, and FAQ lookup into one complete chatbot turn.

**What to expect as output:** The function is defined.

In [ ]:
# SIMULATED AI: In a real system this would use an LLM to generate contextual replies.
# Here, we use intent detection + FAQ lookup.

def customer_support_chatbot(user_message):
    """
    Processes one user message and returns the bot's reply.

    Uses the global memory variables: remembered_user_name
    and conversation_history_list.

    Takes in:
      user_message - the message the user sent (string)

    Returns:
      The bot's reply text (string).
    """

    global remembered_user_name  # we may update this variable inside the function

    # Detect what the user is asking
    detected_intent = detect_intent(user_message)

    # Build the appropriate reply based on the intent
    if detected_intent == 'greeting':
        if remembered_user_name:
            bot_reply = f'Hello again, {remembered_user_name}! How can I help you today?'
        else:
            bot_reply = 'Hello! Welcome to Coffee Corner support. What is your name?'

    elif detected_intent == 'name_introduction':
        # Try to extract the name from the message
        extracted_name = extract_user_name(user_message)
        if extracted_name:
            remembered_user_name = extracted_name  # store name in memory
            bot_reply = f'Nice to meet you, {remembered_user_name}! How can I help you today?'
        else:
            bot_reply = 'I did not quite catch your name. Could you say it again?'

    elif detected_intent == 'farewell':
        name_part = f', {remembered_user_name}' if remembered_user_name else ''
        bot_reply = f'Goodbye{name_part}! Have a wonderful day. Come visit us soon!'

    elif detected_intent in faq_knowledge_base:
        # Look up the answer in the FAQ dictionary
        answer_text = faq_knowledge_base[detected_intent]
        name_prefix = f'{remembered_user_name}, ' if remembered_user_name else ''
        bot_reply = f'{name_prefix}{answer_text}'

    else:
        # Fallback for questions the bot cannot answer
        name_part = f', {remembered_user_name}' if remembered_user_name else ''
        bot_reply = (
            f'I am sorry{name_part}, I did not understand that. '
            'I can help with: hours, location, refunds, contact info, menu, or WiFi. '
            'Which of these would you like to know about?'
        )

    # Record this turn in the conversation history
    conversation_history_list.append({
        'turn': len(conversation_history_list) + 1,
        'user_said': user_message,
        'bot_replied': bot_reply,
        'timestamp': datetime.datetime.now().strftime('%H:%M:%S')
    })

    return bot_reply


print('customer_support_chatbot() defined successfully.')

---
## Step 5: Run the Bot on Sample Messages (Hardcoded Demo)

**What this block does:** We define a list of sample user messages that simulate a full conversation. The bot processes each message in order and we print both the user's message and the bot's reply. Notice that the bot remembers the name after it is introduced.

**Why we need it:** This is the main demo. Running on hardcoded messages ensures the notebook works with 'Run All' and every student sees the same output.

**What to expect as output:** A printed conversation showing all message-reply pairs.

In [ ]:
# Reset memory before running the demo so we start with a clean state
remembered_user_name = None
conversation_history_list.clear()

# These messages simulate a user having a full conversation with the bot
list_of_sample_user_messages = [
    'Hi there!',
    'My name is Layla',
    'What are your opening hours?',
    'And where exactly are you located?',
    'Do you have WiFi?',
    'What food do you serve?',
    'I bought a coffee yesterday and want a refund',
    'How can I reach your team by phone?',
    'What is the weather today?',
    'Thank you, goodbye!'
]

print('=' * 60)
print('CUSTOMER SUPPORT CHATBOT - DEMO CONVERSATION')
print('=' * 60)

for single_user_message in list_of_sample_user_messages:
    print(f'\nUser : {single_user_message}')
    bot_response = customer_support_chatbot(single_user_message)
    print(f'Bot  : {bot_response}')

print('\n' + '=' * 60)
print('End of demo conversation.')

---
## Step 6: Print the Conversation History

**What this block does:** We loop over `conversation_history_list` and print every turn in a formatted log. This shows the memory the bot maintained throughout the conversation.

**Why we need it:** Seeing the conversation log helps students understand how memory accumulates turn by turn. In real chatbots, this history is sent to the LLM as context.

**What to expect as output:** A numbered list of all turns with timestamps.

In [ ]:
print('=' * 60)
print('CONVERSATION HISTORY (BOT MEMORY)')
print('=' * 60)

for history_entry in conversation_history_list:
    print(f"\nTurn {history_entry['turn']} at {history_entry['timestamp']}")
    print(f"  User: {history_entry['user_said']}")
    print(f"  Bot : {history_entry['bot_replied']}")

print('\n' + '=' * 60)
print(f'Total turns stored in memory: {len(conversation_history_list)}')
print(f'User name remembered: {remembered_user_name}')

---
## Step 7: Optional Interactive Version (input)

**What this block does:** This optional cell lets you type real messages to the bot. It uses `input()` to get your message and loops until you type `quit`.

**Important:** This cell is marked optional. Only run it manually - do NOT include it in 'Run All'. In Jupyter, use Kernel > Interrupt to stop it if it gets stuck.

**What to expect:** A real back-and-forth conversation with the bot.

In [ ]:
# ==================================================================
# OPTIONAL INTERACTIVE CELL - run this manually, not with 'Run All'
# Type 'quit' to stop the conversation
# ==================================================================

# Reset memory for a fresh conversation
remembered_user_name = None
conversation_history_list.clear()

print('Coffee Corner Support Bot (type "quit" to exit)')
print('-' * 50)

while True:
    typed_user_message = input('You: ')
    if typed_user_message.lower() == 'quit':
        print('Bot: Goodbye! Have a great day.')
        break
    interactive_bot_reply = customer_support_chatbot(typed_user_message)
    print(f'Bot: {interactive_bot_reply}')

---
## Summary

You built a **Customer Support Chatbot with Memory** that answers FAQs and remembers the user's name across turns.

### Key takeaways
- **Memory with variables:** A single variable (`remembered_user_name`) gives the bot basic personalisation.
- **Conversation history as a list:** Storing turns in a list mirrors how real LLM chatbots work - they send the full history to the model with each new message.
- **Intent detection with keywords:** Simple keyword matching is surprisingly effective for focused domains like customer support.
- **Fallback responses:** Always include a fallback for questions the bot cannot answer. Without one, the bot looks broken.
- **Dictionaries as knowledge bases:** The FAQ dictionary is easy to extend and maintain.

### Connection to the next lab

In **Lab 5**, you will build a Tool-Using Agent. Instead of looking up answers in a dictionary, the agent will call different tools (calculator, weather lookup, clock) depending on what kind of question the user asks.

---

## Try It Yourself

1. **Add a new FAQ topic:** Add a `'parking'` topic to `faq_knowledge_base` and `intent_keyword_map`. Add a sample user message asking about parking and re-run the demo.

2. **Remember more:** Extend the bot's memory to also remember the last topic the user asked about. Use it to add a sentence like 'You were asking about hours earlier.' when the user says goodbye.

3. **Conversation summary:** After the demo runs, write a function called `print_conversation_summary()` that prints only the topics asked about, not the full dialogue.